# 가천대 입학 Agentic RAG 챗봇 (중간평가)
- 근거는 제공된 5개 문서에서만 검색 (웹/외부 조회 없음, 문서 원문 재가공 없음: 로드 후 분할만)
- 구조: HWP/PDF 로드 -> 분할 -> 임베딩(Chroma) + Kiwi BM25 하이브리드 검색 -> `create_agent(rag_tool)`
- **제출용**: 섹션 1~5 + `answer()`. 섹션 6(개발용 평가)은 제출 전에 삭제, API 키는 비우기.

In [ ]:
!pip install -qU langchain langchain-openai langchain-community "langchain-chroma>=0.1.2" "langchain-hwp-hwpx-loader[distribution]" pdfplumber rank_bm25 kiwipiepy pandas

In [ ]:
# 1) 설정 ----------------------------------------------------------------
import os, glob, time, shutil, hashlib, re
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")   # Colab 보안 비밀. 제출 시 코드에 키 없음
except Exception:
    os.environ.setdefault("OPENAI_API_KEY", "")                    # 로컬이면 직접 입력 (제출 전 반드시 비우기)

DOC_DIR = "/content/dataset"        # dataset.zip 압축 해제 폴더 (5개 파일)

CFG = dict(
    chunk_size=800, overlap=100,
    k=4,                    # 벡터 검색 개수
    bm25_k=4,               # BM25 검색 개수 (0이면 BM25 미사용)
    embed_model="text-embedding-3-large",
    agent_model="gpt-5.4-mini",
    max_search=3,           # 에이전트가 검색할 수 있는 최대 횟수
)

SYSTEM_PROMPT = """당신은 가천대학교 2027학년도 입학 안내 챗봇입니다.
- 질문을 받으면 반드시 rag_tool로 제공 문서를 검색한 뒤, 검색된 문맥만 근거로 답변하세요. 사전 지식이나 추측은 금지입니다.
- 검색 결과가 질문에 답하기에 부족하거나 수시/정시/재외국민/지역의사 등 전형이 맞지 않으면, 질의어를 바꿔 다시 검색하세요(최대 {max_search}회).
- 수치, 날짜, 시각, 대상, 예외 조건을 정확히 포함하고, 질문이 묻는 것만 간결하게 답하세요.
- 근거가 끝내 없으면 '제공된 문서에서 확인할 수 없습니다'라고 답하세요.""".format(max_search=CFG["max_search"])

TOOL_DESC = ("가천대학교 2027학년도 입학 자료(재외국민 모집요강, 지역의사선발전형, 수시 모집요강, 정시 모집요강, 수시 면접고사 일정)를 검색합니다. "
             "모집인원, 지원자격, 전형 방법, 일정, 원서접수, 전형료, 제출서류, 수능 반영, 면접 등 입학 관련 모든 질문에 사용하세요. "
             "질의어에는 전형 구분(수시/정시/재외국민/지역의사)과 핵심 키워드를 포함하세요.")

In [ ]:
# 2) 문서 로드 (HWP + PDF) ---------------------------------------------------
from langchain_community.document_loaders import PDFPlumberLoader
from langchain_hwp_hwpx import HwpHwpxLoader

def load_docs():
    docs = []
    for p in sorted(glob.glob(f"{DOC_DIR}/*")):
        name = os.path.basename(p)
        if p.lower().endswith(".hwp"):
            # 배포용 HWP는 pyhwp 백엔드가 필요
            part = HwpHwpxLoader(p, mode="elements", distribution_backend="pyhwp").load()
        elif p.lower().endswith(".pdf"):
            part = PDFPlumberLoader(p).load()
        else:
            continue
        for d in part:
            d.metadata["source"] = name        # 출처 파일명으로 통일
        docs += part
        print(name, len(part))
    return docs

docs = load_docs()

In [ ]:
# 3) 분할 + 임베딩(캐시) + Chroma + Kiwi BM25 -----------------------------------
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_classic.storage import LocalFileStore
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from kiwipiepy import Kiwi

splitter = RecursiveCharacterTextSplitter(chunk_size=CFG["chunk_size"], chunk_overlap=CFG["overlap"],
                                          separators=["\n\n", "\n", " ", ""])
splits = splitter.split_documents(docs)
# 문서 원문은 가공하지 않고 그대로 저장 (출처는 metadata에만 두고, 도구 출력 시에만 표시)
print("chunks:", len(splits))

embedder = CacheBackedEmbeddings.from_bytes_store(OpenAIEmbeddings(model=CFG["embed_model"]), LocalFileStore("./cache/"))
CHROMA_PATH = "./chroma_db"
shutil.rmtree(CHROMA_PATH, ignore_errors=True)       # 재실행 시 중복 적재 방지
db = Chroma.from_documents(splits, embedder, persist_directory=CHROMA_PATH, collection_metadata={"hnsw:space": "cosine"})
vec_retriever = db.as_retriever(search_kwargs={"k": CFG["k"]})

kiwi = Kiwi()
ALLOWED_TAGS = {"NNG", "NNP", "NNB", "VV", "VA", "XR", "SL", "SN"}
def korean_tokenizer(text): return [t.form for t in kiwi.tokenize(text) if t.tag in ALLOWED_TAGS]
bm25_retriever = BM25Retriever.from_documents(splits, preprocess_func=korean_tokenizer)
bm25_retriever.k = CFG["bm25_k"]

In [ ]:
# 4) 검색 도구 + Agentic RAG -----------------------------------------------------
from langchain.tools import tool
from langchain.agents import create_agent

def hybrid_retrieve(query):
    docs = vec_retriever.invoke(query)
    if CFG["bm25_k"] > 0:
        docs = bm25_retriever.invoke(query) + docs
    seen, uniq = set(), []
    for d in docs:
        key = d.page_content.strip()
        if key not in seen:
            seen.add(key); uniq.append(d)
    return uniq

@tool("rag_tool", description=TOOL_DESC)
def rag_tool(query: str) -> str:
    docs = hybrid_retrieve(query)
    if not docs:
        return "<문맥 정보>검색 결과가 없습니다.</문맥 정보>"
    def label(d):
        p = d.metadata.get("page")
        return "[출처: " + d.metadata["source"] + ("" if p is None else " p." + str(p + 1)) + "]\n" + d.page_content
    return "<문맥 정보>" + "\n\n".join(label(d) for d in docs) + "</문맥 정보>"

agent = create_agent(model=CFG["agent_model"], tools=[rag_tool], system_prompt=SYSTEM_PROMPT)

def answer(question: str) -> str:
    """평가용 진입점: 질문 -> 답변 문자열"""
    res = agent.invoke({"messages": [{"role": "user", "content": question}]},
                       config={"recursion_limit": 2 * CFG["max_search"] + 3})
    return res["messages"][-1].content

In [ ]:
# 5) 동작 확인 (공개 샘플 질문) -------------------------------------------------
t = time.perf_counter()
print(answer("가천대학교 대표 전화번호는 무엇인가요?"))
print(f"{time.perf_counter()-t:.1f}s")

---
## 6) 개발용 평가 (제출 전 이 아래 셀 전부 삭제)
`/content/dev_questions.csv`(컬럼: question, answer)를 올려 사용. 채점 공식은 README 그대로: 정답 1.6 + 시간점수(5초 이하 0.4, 초과 1초당 -0.08, 20초 초과는 0점).

In [ ]:
import pandas as pd, numpy as np, json
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

DEV_Q = "/content/dev_questions.csv"
RESULTS = "/content/results.csv"
judge_llm = init_chat_model("gpt-5.4-mini")

def judge(q, gold, pred):
    p = (f"질문: {q}\n예상 정답: {gold}\n모델 답변: {pred}\n\n"
         "표현이 달라도 질문이 요구한 핵심 정보(수치, 날짜, 대상, 예외 조건)가 모두 일치하면 1, 하나라도 빠지거나 다르면 0. 숫자 하나만 출력.")
    return judge_llm.invoke([HumanMessage(p)]).content.strip().startswith("1")

def time_score(t, ok):
    if not ok or t > 20: return 0.0
    return round(1.6 + max(0.0, 0.4 - 0.08 * max(0.0, t - 5)), 3)

LAST = None
def run_eval(name, note=""):
    """현재 CFG/agent로 질문 전체를 1회씩 실행. 시간은 answer() 호출 구간만 측정."""
    global LAST
    qs = pd.read_csv(DEV_Q); rows = []
    for _, r in qs.iterrows():
        t = time.perf_counter()
        try: pred = answer(r.question)
        except Exception as e: pred = f"ERROR {e}"
        dt = time.perf_counter() - t
        ok = (not pred.startswith("ERROR")) and judge(r.question, r.answer, pred)
        rows.append(dict(q=r.question, gold=r.answer, pred=pred, ok=ok, sec=round(dt, 1), score=time_score(dt, ok)))
    LAST = pd.DataFrame(rows)
    n = len(LAST)
    s = dict(name=name, acc=round(LAST.ok.mean(), 3), avg_sec=round(LAST.sec.mean(), 2), p90_sec=round(LAST.sec.quantile(.9), 1),
             score_per_q=round(LAST.score.mean(), 3), est_50q=round(LAST.score.mean() * 50, 1), n=n, note=note,
             **{k: CFG[k] for k in ("chunk_size", "overlap", "k", "bm25_k", "agent_model", "embed_model")})
    pd.DataFrame([s]).to_csv(RESULTS, mode="a", header=not os.path.exists(RESULTS), index=False)
    print(s); return LAST

In [ ]:
# 첫 실험: 베이스라인. 이후 설정(CFG)을 한 항목만 바꾸고 3)~4) 셀을 다시 실행한 뒤 run_eval 실행
df = run_eval("baseline")

In [ ]:
# 결과 비교 / 오답 확인 (오답이 검색 문제인지 생성 문제인지 확인)
display(pd.read_csv(RESULTS))
display(LAST[~LAST.ok][["q", "gold", "pred", "sec"]])
display(LAST.sort_values("sec", ascending=False).head(5)[["q", "sec"]])